# Paper Topics

One tidy row per paper with everything OpenAlex says about its *subject*: the scored topics, the
subfields / fields / domains they roll up to, the primary topic OpenAlex itself picks, and the
work's keywords. Primary key: `paper_id` (OpenAlex `W…`, shared with every other paper output in
`/project/jevans/Dawoon/Science of Science/OpenAlex/output/`).

Sibling of `paper_metadata.ipynb`: that notebook keeps only the primary field / subfield names
(`primary_topic_field`, `primary_topic_subfield`) and the `FoS_*` summary; this one keeps the whole
hierarchy, all topics with their scores, ids as well as names, and the keywords.

## Raw data (directory & structure)
```
/project/jevans/OpenAlex_shared/OpenAlex_2026_Sep_23_flat/   (read-only: official 2026-09-23 release, flattened)
  topics.csv.gz                       id(T…), display_name, subfield_id/_display_name, field_id/_display_name, domain_id/_display_name
  works/works/part_NNNN.parquet       id(W…), publication_year                      -> the row set (one row per work)
  works/topics/part_NNNN.parquet      work_id(W…), topic_id(T…), score              -> up to 3 scored topics per work
  works/works_semantic/part_NNNN.parquet
                                      work_id(URL), primary_topic_json, keywords_json -> primary topic + keywords
```
The 2,040 `part_NNNN` files are partition-aligned (the same works everywhere), so the join is
partition-local. `works_semantic` stores full-URL ids; they are normalised on the way in.

## Output columns
`paper_topics.parquet` — one row per work, multi-valued columns are `;`-joined strings in
**descending score order** (the project convention, cf. `FoS_0`, `author_list`):

| column | meaning |
|---|---|
| `paper_id`, `year` | work id and `publication_year` (same as `paper_metadata.year`) |
| `n_topics` | number of topics attached to the work (0–3) |
| `topic_ids`, `topics`, `topic_scores` | every topic of the work: `T…` ids, display names, scores (4 dp), aligned position-wise |
| `subfield_ids`, `subfields` | **distinct** subfields of those topics, first appearance in score order (252 ASJC-style subfields) |
| `field_ids`, `fields` | distinct fields, same order (26 fields) |
| `domain_ids`, `domains` | distinct domains, same order (4 domains) |
| `primary_topic_id`, `primary_topic`, `primary_topic_score` | `primary_topic` of the work record (`works_semantic.primary_topic_json`) |
| `primary_subfield_id`, `primary_subfield`, `primary_field_id`, `primary_field`, `primary_domain_id`, `primary_domain` | hierarchy of the primary topic (`primary_field` == `paper_metadata.primary_topic_field`) |
| `keyword_ids`, `keywords`, `keyword_scores`, `n_keywords` | the work's keywords (`works_semantic.keywords_json`), score order as stored; null unless `RUN_KEYWORDS` |

`paper_topics_long.parquet` (only if `WRITE_LONG`) — one row per (work, topic):
`paper_id, rank, topic_id, score, topic, subfield_id, subfield, field_id, field, domain_id, domain`,
`rank` = 1 for the highest-scoring topic. This is the shape to join on; the wide table is the
shape to browse.

Ids: topics keep their `T…` accession; subfield / field / domain ids are the bare integer codes
of the OpenAlex URLs (`…/subfields/2507` -> 2507), strings inside the `;`-joined columns and
integers in the `primary_*_id` / long-table columns. The distinct-ness of `subfield_ids` etc. is
by **id**, so a name can repeat inside `subfields` (OpenAlex has two subfields called "Genetics",
1311 under Biochemistry and 2716 under Medicine); the id columns are the ones to group on.
Keyword ids are the full slug after `keywords/`, which may itself contain `/`.

## Run
```
cd "/project/jevans/Dawoon/Science of Science/jobs/OpenAlex"
sbatch --export=ALL,NB=paper_topics -J oa_paper_topics --mem=64G nb.sbatch          # full, ~2-4 h
sbatch --export=ALL,NB=paper_topics,NB_PART_LIMIT=2 -J oa_paper_topics_smoke --mem=16G -t 0:30:00 nb.sbatch
```
`NB_PART_LIMIT=n` runs the first *n* partitions and writes `*.smoke.parquet` next to the real
outputs instead of overwriting them. Nothing here needs the consolidated edge table or the
caches; only `topics.csv.gz` and the three `works/*` datasets are read.


In [1]:
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc
import os, sys, gc, time
import numpy as np, pandas as pd
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/OpenAlex')
import oa_common as oa
ROOT = oa.BASE; OUT = oa.OUT
print('snapshot:', oa.ROOT)

RUN_KEYWORDS = True    # keyword_ids / keywords / keyword_scores from works_semantic.keywords_json
WRITE_LONG   = True    # also write paper_topics_long.parquet (one row per work x topic)
# Smoke test: NB_PART_LIMIT=n -> first n partitions only, outputs get a .smoke suffix.
PART_LIMIT = int(os.environ.get('NB_PART_LIMIT', '0') or 0)
SFX = '.smoke' if PART_LIMIT else ''
OUT_FP      = f'{OUT}/paper_topics{SFX}.parquet'
OUT_LONG_FP = f'{OUT}/paper_topics_long{SFX}.parquet'
OUT_COV_FP  = f'{OUT}/paper_topics_coverage_by_year{SFX}.csv'

WP = oa.parts('works'); TP = oa.parts('topics'); SP = oa.parts('works_semantic')
assert len(WP) == len(TP) == len(SP), 'works, topics and works_semantic partitions must line up'
if PART_LIMIT:
    WP, TP, SP = WP[:PART_LIMIT], TP[:PART_LIMIT], SP[:PART_LIMIT]
    print(f'SMOKE: first {PART_LIMIT} partitions -> {OUT_FP}')
print(f'{len(WP)} partitions | RUN_KEYWORDS={RUN_KEYWORDS} WRITE_LONG={WRITE_LONG}')


snapshot: /project/jevans/OpenAlex_shared/OpenAlex_2026_Sep_23_flat
2040 partitions | RUN_KEYWORDS=True WRITE_LONG=True


## 1. Topic hierarchy lookup (`topics.csv.gz`)

~4,500 topics -> 252 subfields -> 26 fields -> 4 domains. `works/topics` carries only the topic
id and a score; every name and every level above the topic comes from this table.

In [2]:
# 1. topic_id -> names + hierarchy. `LOOK` (arrow) is joined into the per-partition topic rows,
#    where the id codes stay strings so they can be ';'-joined; `LOOK_PD` (pandas, int codes) is
#    reindexed by the primary topic id.
tp = pd.read_csv(f'{oa.ROOT}/topics.csv.gz', dtype=str,
                 usecols=['id', 'display_name', 'subfield_id', 'subfield_display_name',
                          'field_id', 'field_display_name', 'domain_id', 'domain_display_name'])
code_of = lambda s: oa.norm_id(s)              # 'https://openalex.org/subfields/2507' -> '2507'
look = pd.DataFrame({'topic_id':   oa.norm_id(tp['id']),
                     'topic_name': tp['display_name'],
                     'subfield_id': code_of(tp['subfield_id']), 'subfield': tp['subfield_display_name'],
                     'field_id':    code_of(tp['field_id']),    'field':    tp['field_display_name'],
                     'domain_id':   code_of(tp['domain_id']),   'domain':   tp['domain_display_name']})
assert look['topic_id'].is_unique and look.notna().all().all()
for c in ('topic_name', 'subfield', 'field', 'domain'):
    assert not look[c].str.contains(';').any(), f'{c} contains the ";" separator'
LOOK = pa.Table.from_pandas(look, preserve_index=False)
LOOK_PD = look.set_index('topic_id')
for c, dt in (('subfield_id', 'int16'), ('field_id', 'int16'), ('domain_id', 'int8')):
    LOOK_PD[c] = LOOK_PD[c].astype(dt)
print(f"{len(look):,} topics -> {look.subfield_id.nunique()} subfields -> "
      f"{look.field_id.nunique()} fields -> {look.domain_id.nunique()} domains")


4,516 topics -> 252 subfields -> 26 fields -> 4 domains


## 2. One pass over the aligned partitions, streamed to parquet

Per partition: `works/works` gives the row set, `works/topics` x `LOOK` gives the scored topic
rows (sorted by score, ties broken by the snapshot's own order), `works_semantic` gives the primary
topic and the keywords. Lists are built with arrow group-by + `binary_join` (no Python loop over
rows); the three "distinct, in score order" columns come from a (work, id) group-by that keeps
the best score per id. Each partition is written as one row group, so nothing accumulates but
the counters.

In [3]:
%%time
# 2. works x topics x works_semantic, partition by partition -> ParquetWriter
WIDE_SCHEMA = pa.schema([
    ('paper_id', pa.string()), ('year', pa.int32()), ('n_topics', pa.int8()),
    ('topic_ids', pa.string()), ('topics', pa.string()), ('topic_scores', pa.string()),
    ('subfield_ids', pa.string()), ('subfields', pa.string()),
    ('field_ids', pa.string()), ('fields', pa.string()),
    ('domain_ids', pa.string()), ('domains', pa.string()),
    ('primary_topic_id', pa.string()), ('primary_topic', pa.string()), ('primary_topic_score', pa.float64()),
    ('primary_subfield_id', pa.int16()), ('primary_subfield', pa.string()),
    ('primary_field_id', pa.int16()), ('primary_field', pa.string()),
    ('primary_domain_id', pa.int8()), ('primary_domain', pa.string()),
    ('keyword_ids', pa.string()), ('keywords', pa.string()), ('keyword_scores', pa.string()),
    ('n_keywords', pa.int16()),
])
LONG_SCHEMA = pa.schema([
    ('paper_id', pa.string()), ('rank', pa.int8()), ('topic_id', pa.string()), ('score', pa.float64()),
    ('topic', pa.string()), ('subfield_id', pa.int16()), ('subfield', pa.string()),
    ('field_id', pa.int16()), ('field', pa.string()), ('domain_id', pa.int8()), ('domain', pa.string()),
])

def bare(col):
    # 'https://openalex.org/W1' -> 'W1'; bare ids pass through (oa.norm_id, arrow-side).
    return pc.replace_substring_regex(col, pattern=r'^.*/', replacement='')

def joined(tbl, cols):
    # tbl sorted (work_id, score desc) -> DataFrame indexed by paper_id, `cols` ';'-joined in that order.
    g = tbl.group_by('work_id', use_threads=False).aggregate([(c, 'list') for c in cols])
    df = pd.DataFrame({c: pc.binary_join(g[f'{c}_list'], ';').to_pandas() for c in cols})
    df.index = pd.Index(g['work_id'].to_pandas(), name='paper_id')   # set AFTER construction: index= would re-align
    return df

def distinct_ordered(tbl, idcol, namecol):
    # One row per (work, idcol), ordered by the best topic score mapping into it, then snapshot order.
    g = (tbl.group_by(['work_id', idcol], use_threads=False)
            .aggregate([(namecol, 'first'), ('score', 'max'), ('row', 'min')])
            .sort_by([('work_id', 'ascending'), ('score_max', 'descending'), ('row_min', 'ascending')])
            .select(['work_id', idcol, f'{namecol}_first'])
            .rename_columns(['work_id', idcol, namecol]))
    return joined(g, [idcol, namecol])

# primary_topic_json is json.dumps output: {"id": ".../T…", "display_name": "…", "score": s, "subfield": {…}, …}.
# The id sits at the very start and the first "score" is the topic's own (the nested blocks carry none).
PT_ID_RE    = r'^\{"id": "https://openalex\.org/(?P<tid>T\d+)"'
PT_SCORE_RE = r'"score": (?P<score>[0-9.eE+-]+)'
PRIMARY_COLS = [('topic_name', 'primary_topic'), ('subfield_id', 'primary_subfield_id'), ('subfield', 'primary_subfield'),
                ('field_id', 'primary_field_id'), ('field', 'primary_field'),
                ('domain_id', 'primary_domain_id'), ('domain', 'primary_domain')]

def primary_block(s):
    ex_id = pc.extract_regex(s['primary_topic_json'], PT_ID_RE)
    ex_sc = pc.extract_regex(s['primary_topic_json'], PT_SCORE_RE)
    df = pd.DataFrame({'primary_topic_id':    pc.struct_field(ex_id, 'tid').to_pandas(),
                       'primary_topic_score': pc.cast(pc.struct_field(ex_sc, 'score'), pa.float64()).to_pandas()})
    df.index = pd.Index(bare(s['work_id']).to_pandas(), name='paper_id')
    m = LOOK_PD.reindex(df['primary_topic_id'])
    for src, dst in PRIMARY_COLS:
        df[dst] = m[src].to_numpy()
    return df

# keywords_json is a json.dumps list of {"id": ".../keywords/slug", "display_name": "…", "score": s}.
# One RE2 rewrite per output column turns the whole list into 'v;v;v;' in place (no json.loads on
# 372M rows); a row that still contains '{' afterwards did not parse and is nulled.
KW_ITEM_RE = (r'\{"id": "https://openalex\.org/keywords/([^"]*)", '
              r'"display_name": "((?:[^"\\]|\\.)*)", "score": ([0-9.eE+-]+)\}(?:, )?')

def keywords_block(s):
    kj = s['keywords_json']
    def pull(ref):
        x = pc.replace_substring_regex(kj, pattern=KW_ITEM_RE, replacement=ref)      # items -> 'v;v;v;'
        return pc.replace_substring_regex(x, pattern=r'^\[(.*);\]$', replacement=r'\1')  # drop the array brackets
    ids, names, scores = pull(r'\1;'), pull(r'\2;'), pull(r'\3;')
    names = pc.replace_substring(names, pattern='\\"', replacement='"')               # un-escape quotes in names
    good = pc.and_(pc.invert(pc.match_substring(ids, '{')), pc.invert(pc.match_substring(ids, '[')))
    null = pa.scalar(None, pa.string())
    n = pc.if_else(good, pc.add(pc.count_substring(ids, ';'), 1), 0)
    bad = int(pc.sum(pc.and_(pc.invert(good), pc.match_substring(ids, '{'))).as_py() or 0)
    df = pd.DataFrame({'keyword_ids':    pc.if_else(good, ids, null).to_pandas(),
                       'keywords':       pc.if_else(good, names, null).to_pandas(),
                       'keyword_scores': pc.if_else(good, scores, null).to_pandas(),
                       'n_keywords':     n.to_pandas().fillna(0).astype('int16')})
    df.index = pd.Index(bare(s['work_id']).to_pandas(), name='paper_id')
    return df, bad

WIDE_COLS = WIDE_SCHEMA.names
stats = dict(works=0, with_topics=0, with_primary=0, orphans=0, primary_vs_top_both=0, primary_vs_top_agree=0,
             unknown_topic_rows=0, keyword_parse_failures=0, long_rows=0)
n_topics_dist = pd.Series(dtype='int64'); field_counts = pd.Series(dtype='int64')
cov = pd.DataFrame(columns=['works', 'with_topics'], dtype='int64')

for p in (OUT_FP, OUT_LONG_FP):
    if os.path.exists(p):
        os.remove(p)
wide_w = pq.ParquetWriter(OUT_FP, WIDE_SCHEMA, compression='snappy')
long_w = pq.ParquetWriter(OUT_LONG_FP, LONG_SCHEMA, compression='snappy') if WRITE_LONG else None
t0 = time.time()
for i, (fw, ft, fs) in enumerate(zip(WP, TP, SP)):
    # -- row set
    w = pq.read_table(fw, columns=['id', 'publication_year'])
    out = pd.DataFrame({'year': w['publication_year'].to_pandas().astype('Int32')})
    out.index = pd.Index(bare(w['id']).to_pandas(), name='paper_id')
    # -- scored topics, joined to the hierarchy, best first
    t = pq.read_table(ft, columns=['work_id', 'topic_id', 'score'])
    t = (t.set_column(0, 'work_id', bare(t['work_id']))
          .append_column('row', pa.array(np.arange(t.num_rows, dtype=np.int64)))
          .join(LOOK, keys='topic_id', join_type='left outer'))
    unknown = int(pc.sum(pc.is_null(t['topic_name'])).as_py() or 0)
    if unknown:
        stats['unknown_topic_rows'] += unknown
        t = t.filter(pc.is_valid(t['topic_name']))
    t = t.sort_by([('work_id', 'ascending'), ('score', 'descending'), ('row', 'ascending')])
    t = t.append_column('score_s', pc.cast(pc.round(t['score'], 4), pa.string()))
    tw = joined(t, ['topic_id', 'topic_name', 'score_s']).rename(
        columns={'topic_id': 'topic_ids', 'topic_name': 'topics', 'score_s': 'topic_scores'})
    sf = distinct_ordered(t, 'subfield_id', 'subfield').rename(columns={'subfield_id': 'subfield_ids', 'subfield': 'subfields'})
    fd = distinct_ordered(t, 'field_id', 'field').rename(columns={'field_id': 'field_ids', 'field': 'fields'})
    dm = distinct_ordered(t, 'domain_id', 'domain').rename(columns={'domain_id': 'domain_ids', 'domain': 'domains'})
    stats['orphans'] += int((~tw.index.isin(out.index)).sum())     # topic rows whose work is not in this partition
    # -- primary topic + keywords
    s = pq.read_table(fs, columns=['work_id', 'primary_topic_json'] + (['keywords_json'] if RUN_KEYWORDS else []))
    out = out.join(tw).join(sf).join(fd).join(dm).join(primary_block(s))
    if RUN_KEYWORDS:
        kw, bad = keywords_block(s)
        out = out.join(kw); stats['keyword_parse_failures'] += bad
    else:
        out['keyword_ids'] = None; out['keywords'] = None; out['keyword_scores'] = None; out['n_keywords'] = 0
    out['n_topics'] = (out['topic_ids'].str.count(';').fillna(-1) + 1).astype('int8')   # no topics -> 0
    out['n_keywords'] = out['n_keywords'].fillna(0).astype('int16')
    for c, dt in (('primary_subfield_id', 'Int16'), ('primary_field_id', 'Int16'), ('primary_domain_id', 'Int8')):
        out[c] = out[c].astype(dt)
    # -- counters
    stats['works'] += len(out)
    stats['with_topics'] += int((out['n_topics'] > 0).sum())
    stats['with_primary'] += int(out['primary_topic_id'].notna().sum())
    both = out['primary_topic_id'].notna() & out['topic_ids'].notna()
    stats['primary_vs_top_both'] += int(both.sum())
    stats['primary_vs_top_agree'] += int((out.loc[both, 'primary_topic_id']
                                          == out.loc[both, 'topic_ids'].str.split(';').str[0]).sum())
    n_topics_dist = n_topics_dist.add(out['n_topics'].value_counts(), fill_value=0)
    field_counts = field_counts.add(out['primary_field'].value_counts(), fill_value=0)
    cov = cov.add(pd.DataFrame({'works': out.groupby('year').size(),
                                'with_topics': out.groupby('year')['n_topics'].apply(lambda x: int((x > 0).sum()))}),
                  fill_value=0)
    # -- write
    wide_w.write_table(pa.Table.from_pandas(out.reset_index()[WIDE_COLS], schema=WIDE_SCHEMA, preserve_index=False))
    if WRITE_LONG:
        lg = t.select(['work_id', 'topic_id', 'score', 'topic_name', 'subfield_id', 'subfield',
                       'field_id', 'field', 'domain_id', 'domain']).to_pandas()
        lg.insert(1, 'rank', (lg.groupby('work_id', sort=False).cumcount() + 1).astype('int8'))
        lg = lg.rename(columns={'work_id': 'paper_id', 'topic_name': 'topic'})
        for c, dt in (('subfield_id', 'int16'), ('field_id', 'int16'), ('domain_id', 'int8')):
            lg[c] = lg[c].astype(dt)
        long_w.write_table(pa.Table.from_pandas(lg, schema=LONG_SCHEMA, preserve_index=False))
        stats['long_rows'] += len(lg); del lg
    del w, t, tw, sf, fd, dm, s, out
    if (i + 1) % 100 == 0 or i + 1 == len(WP):
        el = time.time() - t0
        print(f'  {i+1}/{len(WP)} partitions | {stats["works"]:,} works | {el/60:.1f} min '
              f'(eta {el/(i+1)*(len(WP)-i-1)/60:.0f} min)', flush=True)
        gc.collect()
wide_w.close()
if long_w:
    long_w.close()
print(f'WROTE {OUT_FP}  ({stats["works"]:,} rows, {len(WIDE_COLS)} cols, {os.path.getsize(OUT_FP)/1e9:.2f} GB)')
if WRITE_LONG:
    print(f'WROTE {OUT_LONG_FP}  ({stats["long_rows"]:,} rows, {os.path.getsize(OUT_LONG_FP)/1e9:.2f} GB)')


  100/2040 partitions | 1,861 works | 0.1 min (eta 1 min)
  200/2040 partitions | 19,266,741 works | 13.5 min (eta 125 min)
  300/2040 partitions | 41,667,642 works | 28.9 min (eta 168 min)
  400/2040 partitions | 68,822,754 works | 48.4 min (eta 198 min)
  500/2040 partitions | 74,661,353 works | 52.9 min (eta 163 min)
  600/2040 partitions | 83,075,603 works | 59.5 min (eta 143 min)
  700/2040 partitions | 96,478,988 works | 69.7 min (eta 133 min)
  800/2040 partitions | 124,017,819 works | 85.0 min (eta 132 min)
  900/2040 partitions | 147,596,889 works | 98.6 min (eta 125 min)
  1000/2040 partitions | 175,534,881 works | 122.4 min (eta 127 min)
  1100/2040 partitions | 205,754,118 works | 137.1 min (eta 117 min)
  1200/2040 partitions | 237,149,161 works | 155.7 min (eta 109 min)
  1300/2040 partitions | 275,547,032 works | 189.4 min (eta 108 min)
  1400/2040 partitions | 297,333,882 works | 205.9 min (eta 94 min)
  1500/2040 partitions | 321,763,591 works | 225.1 min (eta 81 min)


## 3. Validation

Coverage, the topic-count distribution, and two consistency checks: every topic row must belong
to the partition it sits in (`orphans == 0`, i.e. the alignment the pass relies on), and the
primary topic of the work record must be the highest-scoring row of `works/topics` (OpenAlex
defines it that way, so anything below ~100% would mean the two datasets are from different
snapshots).

In [4]:
# 3. Summary from the counters (nothing is read back in full)
W = stats['works']
print(f"works {W:,} | with topics {stats['with_topics']/W*100:.1f}% | with primary topic "
      f"{stats['with_primary']/W*100:.1f}% | topic rows written {stats['long_rows']:,}")
print('n_topics distribution:', {int(k): f'{v/W*100:.1f}%' for k, v in n_topics_dist.sort_index().items()})
print(f"primary_topic_id == top-scored works/topics row: "
      f"{stats['primary_vs_top_agree']/max(stats['primary_vs_top_both'],1)*100:.3f}%  (n={stats['primary_vs_top_both']:,})")
print(f"orphan topic rows (work not in its partition): {stats['orphans']:,} | "
      f"topic ids missing from topics.csv.gz: {stats['unknown_topic_rows']:,} | "
      f"keyword_json parse failures: {stats['keyword_parse_failures']:,}")
assert stats['orphans'] == 0, 'partition alignment broken: a topic row landed outside its work partition'
print('\ntop primary fields:')
print((field_counts.sort_values(ascending=False).head(10) / stats['with_primary'] * 100).round(1).to_string())
cov = cov.astype('int64'); cov['coverage_pct'] = (cov['with_topics'] / cov['works'] * 100).round(1)
cov.index = cov.index.astype('Int64'); cov.index.name = 'year'
cov.sort_index().to_csv(OUT_COV_FP)
print(f'\ncoverage by year -> {OUT_COV_FP}')
print(cov.loc[[y for y in cov.index if pd.notna(y) and 1990 <= y <= 2025]].sort_index().to_string())
pf = pq.ParquetFile(OUT_FP)
print(f'\n{OUT_FP}: {pf.metadata.num_rows:,} rows, {pf.metadata.num_row_groups} row groups')
print(pf.schema_arrow)


works 476,196,327 | with topics 79.4% | with primary topic 79.4% | topic rows written 1,001,399,607
n_topics distribution: {0: '20.6%', 1: '10.6%', 2: '6.8%', 3: '62.0%'}
primary_topic_id == top-scored works/topics row: 100.000%  (n=378,173,667)
orphan topic rows (work not in its partition): 0 | topic ids missing from topics.csv.gz: 0 | keyword_json parse failures: 0

top primary fields:
primary_field
Engineering                                     15.7
Social Sciences                                 14.4
Medicine                                        13.7
Arts and Humanities                              7.3
Computer Science                                 6.0
Agricultural and Biological Sciences             5.4
Physics and Astronomy                            5.1
Biochemistry, Genetics and Molecular Biology     5.1
Environmental Science                            4.6
Economics, Econometrics and Finance              3.5

coverage by year -> /project/jevans/Dawoon/Science of Science/Op

## 4. Example

In [5]:
# Ten rows via iter_batches: the parquet is read incrementally, never materialised in full.
pd.set_option('display.max_colwidth', 60); pd.set_option('display.width', 250)
ex = next(pq.ParquetFile(OUT_FP).iter_batches(batch_size=10)).to_pandas()
display(ex[['paper_id', 'year', 'n_topics', 'topics', 'topic_scores', 'subfields', 'fields', 'domains']])
display(ex[['paper_id', 'primary_topic', 'primary_topic_score', 'primary_subfield', 'primary_field',
            'primary_domain', 'n_keywords', 'keywords']])
if WRITE_LONG:
    display(next(pq.ParquetFile(OUT_LONG_FP).iter_batches(batch_size=10)).to_pandas())


,paper_id,year,n_topics,topics,topic_scores,subfields,fields,domains
0,W414088417,1989,3,"Hermeneutics and Narrative Identity;Aging, Elder Care, a...",0.9879;0.9749;0.9566,Philosophy;General Health Professions,Arts and Humanities;Health Professions,Social Sciences;Health Sciences
1,W416029425,1990,3,"Hermeneutics and Narrative Identity;Aging, Elder Care, a...",0.9879;0.9749;0.9566,Philosophy;General Health Professions,Arts and Humanities;Health Professions,Social Sciences;Health Sciences
2,W398685085,1930,3,"Hermeneutics and Narrative Identity;Aging, Elder Care, a...",0.9879;0.9749;0.9566,Philosophy;General Health Professions,Arts and Humanities;Health Professions,Social Sciences;Health Sciences
3,W412110403,1965,3,"Hermeneutics and Narrative Identity;Aging, Elder Care, a...",0.9879;0.9749;0.9566,Philosophy;General Health Professions,Arts and Humanities;Health Professions,Social Sciences;Health Sciences
4,W591526923,1962,3,"Hermeneutics and Narrative Identity;Aging, Elder Care, a...",0.9879;0.9749;0.9566,Philosophy;General Health Professions,Arts and Humanities;Health Professions,Social Sciences;Health Sciences
5,W151270144,1970,3,"Hermeneutics and Narrative Identity;Aging, Elder Care, a...",0.9879;0.9749;0.9566,Philosophy;General Health Professions,Arts and Humanities;Health Professions,Social Sciences;Health Sciences
6,W144693669,1993,3,"Hermeneutics and Narrative Identity;Aging, Elder Care, a...",0.9879;0.9749;0.9566,Philosophy;General Health Professions,Arts and Humanities;Health Professions,Social Sciences;Health Sciences
7,W146474571,2003,3,Management and Optimization Techniques;Occupational Heal...,0.5294;0.2077;0.0186,Strategy and Management;Occupational Therapy;Materials C...,"Business, Management and Accounting;Health Professions;M...",Social Sciences;Health Sciences;Physical Sciences
8,W579042073,1967,3,"Hermeneutics and Narrative Identity;Aging, Elder Care, a...",0.9879;0.9749;0.9566,Philosophy;General Health Professions,Arts and Humanities;Health Professions,Social Sciences;Health Sciences
9,W237726659,1968,3,"Hermeneutics and Narrative Identity;Aging, Elder Care, a...",0.9879;0.9749;0.9566,Philosophy;General Health Professions,Arts and Humanities;Health Professions,Social Sciences;Health Sciences


,paper_id,primary_topic,primary_topic_score,primary_subfield,primary_field,primary_domain,n_keywords,keywords
0,W414088417,Hermeneutics and Narrative Identity,0.9879,Philosophy,Arts and Humanities,Social Sciences,1,Computer science
1,W416029425,Hermeneutics and Narrative Identity,0.9879,Philosophy,Arts and Humanities,Social Sciences,3,Theology;Philosophy;Political science
2,W398685085,Hermeneutics and Narrative Identity,0.9879,Philosophy,Arts and Humanities,Social Sciences,1,Political science
3,W412110403,Hermeneutics and Narrative Identity,0.9879,Philosophy,Arts and Humanities,Social Sciences,1,Political science
4,W591526923,Hermeneutics and Narrative Identity,0.9879,Philosophy,Arts and Humanities,Social Sciences,1,Philosophy
5,W151270144,Hermeneutics and Narrative Identity,0.9879,Philosophy,Arts and Humanities,Social Sciences,2,Psychology;Medicine
6,W144693669,Hermeneutics and Narrative Identity,0.9879,Philosophy,Arts and Humanities,Social Sciences,2,Philosophy;Political science
7,W146474571,Management and Optimization Techniques,0.5294,Strategy and Management,"Business, Management and Accounting",Social Sciences,5,Physics;Humanities;Material handling;Engineering;Art
8,W579042073,Hermeneutics and Narrative Identity,0.9879,Philosophy,Arts and Humanities,Social Sciences,1,Political science
9,W237726659,Hermeneutics and Narrative Identity,0.9879,Philosophy,Arts and Humanities,Social Sciences,1,Philosophy


,paper_id,rank,topic_id,score,topic,subfield_id,subfield,field_id,field,domain_id,domain
0,W1000329410,1,T13497,0.9879,Hermeneutics and Narrative Identity,1211,Philosophy,12,Arts and Humanities,2,Social Sciences
1,W1000329410,2,T13695,0.9749,"Aging, Elder Care, and Social Issues",3600,General Health Professions,36,Health Professions,4,Health Sciences
2,W1000329410,3,T13099,0.9566,"Health, Medicine and Society",3600,General Health Professions,36,Health Professions,4,Health Sciences
3,W1001888705,1,T13497,0.9879,Hermeneutics and Narrative Identity,1211,Philosophy,12,Arts and Humanities,2,Social Sciences
4,W1001888705,2,T13695,0.9749,"Aging, Elder Care, and Social Issues",3600,General Health Professions,36,Health Professions,4,Health Sciences
5,W1001888705,3,T13099,0.9566,"Health, Medicine and Society",3600,General Health Professions,36,Health Professions,4,Health Sciences
6,W1004167937,1,T13497,0.9879,Hermeneutics and Narrative Identity,1211,Philosophy,12,Arts and Humanities,2,Social Sciences
7,W1004167937,2,T13695,0.9749,"Aging, Elder Care, and Social Issues",3600,General Health Professions,36,Health Professions,4,Health Sciences
8,W1004167937,3,T13099,0.9566,"Health, Medicine and Society",3600,General Health Professions,36,Health Professions,4,Health Sciences
9,W1006804087,1,T13497,0.9879,Hermeneutics and Narrative Identity,1211,Philosophy,12,Arts and Humanities,2,Social Sciences
